In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("All libraries imported successfully.")

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


# text-Cleaning
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = nltk.word_tokenize(text)
    clean_words = [stemmer.stem(word) for word in words if word not in stop_words]
    return " ".join(clean_words)

print("Text-cleaning function is ready.")

try:

    df = pd.read_csv('tmdb_5000_movies.csv', on_bad_lines='skip', engine='python')

    print(f"Successfully loaded {len(df)} movies.")

    df.dropna(subset=['overview'], inplace=True)
    print(f"Down to {len(df)} movies after removing those with no plot overview.")

    print("Cleaning all movie overviews... (This may take a moment)")
    df['clean_overview'] = df['overview'].apply(preprocess_text)
    print("All overviews have been cleaned and are ready for ML.")

    # TF-IDF and K-Means
    vectorizer = TfidfVectorizer(max_features=2000) #TF-IDF (Term Frequency-Inverse Document Frequency) vectors are numerical representations of text used in natural language processing (NLP) to measure the importance of a word in a document within a larger collection of documents
    print("\nCreating TF-IDF 'fingerprints' for all movies...")
    tfidf_matrix = vectorizer.fit_transform(df['clean_overview'])
    print("Fingerprints created.")

    NUMBER_OF_CLUSTERS = 20
    kmeans = KMeans(n_clusters=NUMBER_OF_CLUSTERS, random_state=42, n_init=10)
    print(f"Grouping movies into {NUMBER_OF_CLUSTERS} clusters...")
    kmeans.fit(tfidf_matrix)
    print("Clustering complete.")

    # Save Results
    df['cluster'] = kmeans.labels_

    # Model evaluation
    silhouette_avg = silhouette_score(tfidf_matrix, kmeans.labels_)
    print(f"\n--- MODEL EVALUATION SCORE ---")
    print(f"The Silhouette Score for our {NUMBER_OF_CLUSTERS} clusters is: {silhouette_avg:.4f}")

    #clustered movies
    print("\n--- HERE ARE YOUR MOVIE 'PERSONALITIES': ---")
    feature_names = vectorizer.get_feature_names_out()
    cluster_centers = kmeans.cluster_centers_
    order_centroids = cluster_centers.argsort()[:, ::-1]

    for i in range(NUMBER_OF_CLUSTERS):
        print(f"\n========== CLUSTER {i} ('Personality #{i}') ==========")
        top_keywords = [feature_names[ind] for ind in order_centroids[i, :10]]
        print(f"TOP KEYWORDS: {', '.join(top_keywords)}")
        sample_movies = df[df['cluster'] == i]['title'].head(5).tolist()
        print("SAMPLE MOVIES:")
        for movie in sample_movies:
            print(f"  - {movie}")

    df.to_csv('movies_with_clusters.csv', index=False)
    print("\nSuccessfully saved the results to 'movies_with_clusters.csv'")

except FileNotFoundError:
    print("\n--- ERROR! ---")
    print("File 'tmdb_5000_movies.csv' not found.")
except Exception as e:
    print(f"\nAn error occurred: {e}")

All libraries imported successfully.
NLTK helpers downloaded.
Text-cleaning function is ready.
Successfully loaded 4803 movies.
Down to 4800 movies after removing those with no plot overview.
Cleaning all movie overviews... (This may take a moment)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


All overviews have been cleaned and are ready for ML.

Creating TF-IDF 'fingerprints' for all movies...
Fingerprints created.
Grouping movies into 20 clusters...
Clustering complete.

--- MODEL EVALUATION SCORE ---
The Silhouette Score for our 20 clusters is: 0.0073
(A score close to +1 is good, a score close to 0 means the clusters overlap)

--- HERE ARE YOUR MOVIE 'PERSONALITIES': ---

========== CLUSTER 0 ('Personality #0') ==========
TOP KEYWORDS: earth, planet, alien, space, crew, human, race, mission, find, astronaut
SAMPLE MOVIES:
  - Avatar
  - Pirates of the Caribbean: At World's End
  - John Carter
  - Avengers: Age of Ultron
  - Battleship

========== CLUSTER 1 ('Personality #1') ==========
TOP KEYWORDS: get, friend, one, find, coupl, year, go, back, come, discov
SAMPLE MOVIES:
  - Spider-Man 3
  - Harry Potter and the Half-Blood Prince
  - The Amazing Spider-Man
  - The Hobbit: The Desolation of Smaug
  - Spider-Man 2

========== CLUSTER 2 ('Personality #2') ==========
TOP 